# Solutions · 03 · Banded Newton and the cost of implicit steps

Worked solutions to the exercises of [notebook 03](../notebooks/03_banded_newton_and_the_cost_of_implicit_steps.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, linalg as la, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Measure the Jacobian evaluations and Newton iterations of `problems.porous_medium()` for BDF1, BDF2 and BDF4 at the same step. Which order is cheapest per unit of accuracy against the Barenblatt solution?

In [2]:
p = pr.porous_medium()
for order in (1, 2, 4):
    s = p.solve("bdf", 2e-3, order=order)
    print(f"BDF{order}: error vs Barenblatt {p.error(s):.2e}, Newton {s.newton_iterations} ({s.newton_iterations / s.steps:.1f}/step), Jacobians {s.jacobian_evaluations}")

BDF1: error vs Barenblatt 1.28e-03, Newton 3648 (7.3/step), Jacobians 125


BDF2: error vs Barenblatt 4.90e-04, Newton 3522 (7.0/step), Jacobians 89


BDF4: error vs Barenblatt 4.72e-04, Newton 3439 (6.9/step), Jacobians 69


At the same step the higher orders cost about the same number of Newton iterations and give similar errors:
on this problem the error is dominated by the spatial discretisation of the front, and the time order barely
matters. BDF2 is the practical choice - order enough, the smallest start-up cost.

## Exercise 2

For a 2-D grid of n x n points the Jacobian band is n. Tabulate the banded LU cost 2 N b^2 (N = n^2) and the dense cost 2 N^3/3 for n = 16, 32, 64, 128, and the memory of the banded factor (2 b + 1) N numbers versus N^2.

In [3]:
rows = []
for n in (16, 32, 64, 128):
    N, b = n * n, n
    rows.append((n, N, 2 * N * b**2, 2 * N**3 / 3, (2 * b + 1) * N * 8 / 1e6, N * N * 8 / 1e9))
print(pd.DataFrame(rows, columns=["n", "N = n^2", "banded LU flops", "dense LU flops", "banded factor (MB)", "dense (GB)"]).to_string(index=False, float_format="%.3g"))

  n  N = n^2  banded LU flops  dense LU flops  banded factor (MB)  dense (GB)
 16      256           131072        1.12e+07              0.0676    0.000524
 32     1024          2097152        7.16e+08               0.532     0.00839
 64     4096         33554432        4.58e+10                4.23       0.134
128    16384        536870912        2.93e+12                33.7        2.15


The banded factorisation of an n × n grid's Jacobian costs 2n⁴ flops and stores (2n + 1)n² numbers; the
dense one costs 2n⁶/3 and stores n⁴. At n = 128 that is 10⁹ flops and 33 MB against 3×10¹² flops and 2 GB -
the difference between an interactive computation and an impossible one.

## Exercise 3

**Implement it yourself:** write a banded LU without pivoting for a tridiagonal matrix (the Thomas algorithm) and use it inside a backward-Euler step of the heat equation; compare the step with `integrators.bdf(order=1)`.

In [4]:
def thomas(a, b, c, d):                                    # sub-, main, super-diagonal, right-hand side
    n = len(d); cp, dp = np.zeros(n), np.zeros(n)
    cp[0], dp[0] = c[0] / b[0], d[0] / b[0]
    for i in range(1, n):
        m = b[i] - a[i] * cp[i - 1]
        cp[i] = c[i] / m if i < n - 1 else 0.0
        dp[i] = (d[i] - a[i] * dp[i - 1]) / m
    x = np.zeros(n); x[-1] = dp[-1]
    for i in range(n - 2, -1, -1):
        x[i] = dp[i] - cp[i] * x[i + 1]
    return x
g = grids.uniform(0, 1, 40); h = 0.005
u0 = np.sin(np.pi * g.x)
r = h / g.dx**2
a = np.r_[0.0, -r * np.ones(39), 0.0]; b = np.r_[1.0, (1 + 2 * r) * np.ones(39), 1.0]; c = np.r_[0.0, -r * np.ones(39), 0.0]
u = thomas(a, b, c, u0)                                     # one backward-Euler step
lib = ti.bdf(lambda t, v: op.diffusion(v, g, 1.0, bc.Dirichlet(0.0), bc.Dirichlet(0.0), t), u0, (0, h), h, 1, band=(1, 1), n_out=1).y[-1]
print("Thomas step vs library backward Euler:", f"{np.max(np.abs(u - lib)):.1e}")

Thomas step vs library backward Euler: 4.4e-16


The Thomas algorithm is banded LU without pivoting for a tridiagonal matrix: forward elimination of the
sub-diagonal and back substitution, 8n flops. The library's step uses pivoted banded LU through a Newton
iteration that converges in one step for this linear problem; the two agree to rounding.